# Week 18: Autonomous AI Agents & Multi-Agent Systems
## Notebook 02: Multi-Agent Collaboration & Adversarial Debate

### 1. Conceptual Intuition & Real-World Use Case
Single LLMs suffer from blind spots: once a model makes an assumption in its output, its autoregressive self-attention attends to its own prior tokens, reinforcing its hallucinations.

**Multi-Agent Systems (MAS)** break this failure loop by assigning distinct specialized personas to separate agents:
- **Coder Agent:** Writes the initial code implementation.
- **Security & QA Critic Agent:** Adversarially probes the code for buffer overflows, race conditions, and missing edge cases.
- **Coordinator / Judge Agent:** Synthesizes feedback and enforces termination criteria.

Research shows that **Multi-Agent Debate** significantly boosts factual accuracy and coding benchmark scores (SWE-bench, HumanEval).

---

### 2. Mathematical Foundations: Multi-Agent Consensus
Let $A_1, A_2, \dots, A_M$ be $M$ distinct agent models with independent system prompts.
For input query $Q$ over debate rounds $t = 1, \dots, T$:
$$R_i^{(t)} = f_{A_i}\left(Q, \{R_j^{(t-1)}\}_{j \neq i}\right)$$
At each round, Agent $i$ conditions on the critique of its peers, updating its beliefs until convergence or consensus:
$$\text{Consensus}(R^{(T)}) \ge 1 - \epsilon$$

In [ ]:
class Agent:
    def __init__(self, name, role_prompt):
        self.name = name
        self.role_prompt = role_prompt

    def respond(self, context):
        raise NotImplementedError

# Specialized Mock Agent Implementations
class DeveloperAgent(Agent):
    def generate_solution(self, task):
        return {
            "code": "def divide_numbers(a, b):\n    return a / b",
            "explanation": "Simple division of two numbers."
        }

    def refine(self, task, critique):
        return {
            "code": "def divide_numbers(a, b):\n    if b == 0:\n        raise ValueError('Division by zero is undefined')\n    return a / b",
            "explanation": "Added zero division validation based on critic review."
        }

class SecurityCriticAgent(Agent):
    def review(self, task, code):
        issues = []
        if "b == 0" not in code and "/ b" in code:
            issues.append("Vulnerability: ZeroDivisionError uncaught when b=0.")
        if "raise ValueError" not in code:
            issues.append("Missing explicit exception handling.")
        return {
            "passed": len(issues) == 0,
            "issues": issues
        }

# Step 2: Multi-Agent Orchestration Loop
def run_multi_agent_workflow(task):
    dev = DeveloperAgent("DevBot", "Senior Python Engineer")
    critic = SecurityCriticAgent("SecBot", "Security Auditor")
    
    print(f"Task: {task}")
    print("\n[Round 1] Developer creates initial draft...")
    draft = dev.generate_solution(task)
    print("Initial Code:\n", draft["code"])
    
    print("\n[Round 1] Security Critic audits code...")
    review = critic.review(task, draft["code"])
    print(f"Audit Passed: {review['passed']}")
    print("Found Issues:", review["issues"])
    
    if not review["passed"]:
        print("\n[Round 2] Developer refines code addressing critique...")
        refined = dev.refine(task, review["issues"])
        print("Refined Code:\n", refined["code"])
        
        final_review = critic.review(task, refined["code"])
        print(f"\nFinal Audit Passed: {final_review['passed']}")
        return refined["code"]
    return draft["code"]

task = "Write a function to safely divide two numeric arguments."
approved_code = run_multi_agent_workflow(task)

### 3. Step-by-Step Code Walkthrough
1. **Role Specialization:** Developer prioritizes feature delivery, while Critic prioritizes defensive security and edge cases.
2. **Structured Feedback:** The critic returns programmatic issues rather than generic text.
3. **Convergence Loop:** Developer consumes the critique to fix the flaw, passing the audit on Round 2.

---

### 4. Industry Applications & Trade-offs
- **Autonomous Software Engineering (e.g. Devin, GitHub Workspace):** Agents pair-program to resolve complex GitHub issues end-to-end.
- **Trade-off:** Token consumption and API latency scale linearly with the number of agents and debate rounds.

---

### 5. Top Interview Questions & Answers
**Q1: How do you prevent multi-agent systems from getting stuck in infinite debate loops?**
> **Answer:** Implement strict termination constraints: (1) Maximum round limit $T_{max}$, (2) Deterministic convergence metric (e.g. linter or test suite passing), and (3) A tie-breaking Judge / Orchestrator agent with authority to force a conclusion.

**Q2: What is the "Groupthink" or sycophancy risk in multi-agent collaboration?**
> **Answer:** If agents have access to each other's full conversation history, weaker agents often prematurely agree with the majority opinion or defer to the loudest agent. Mitigate this by enforcing **blind initial generation** before sharing peer outputs.